In [1]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 12.6 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/CV_PROJECT/data_set"

Mounted at /content/drive


In [3]:
!wget -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task


--2026-03-31 03:30:19--  https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.121.207, 192.178.142.207, 142.251.108.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.121.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7819105 (7.5M) [application/octet-stream]
Saving to: ‘hand_landmarker.task’

hand_landmarker.tas 100%[===================>]   7.46M  --.-KB/s    in 0.09s   

2026-03-31 03:30:20 (79.6 MB/s) - ‘hand_landmarker.task’ saved [7819105/7819105]



In [7]:
import cv2
import mediapipe as mp
import os
import numpy as np
import sys

# Add the directory containing utils.py to the Python path
sys.path.append('/content/drive/MyDrive/CV_PROJECT/')

import utils

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Load model
base_options = python.BaseOptions(model_asset_path="hand_landmarker.task")
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1
)

detector = vision.HandLandmarker.create_from_options(options)


def get_hand_landmarks(image):
    # convert to RGB
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)

    result = detector.detect(mp_image)

    if result.hand_landmarks:
        hand = result.hand_landmarks[0]

        features = []
        for lm in hand:
            features.extend([lm.x, lm.y, lm.z])

        return features
    return None

X = []
y = []

print("Folders:", os.listdir(DATA_DIR))

for label in os.listdir(DATA_DIR):
    label_path = os.path.join(DATA_DIR, label)

    if not os.path.isdir(label_path):
        continue

    print(label, "->", len(os.listdir(label_path)))

    for img_name in os.listdir(label_path):
        img_path = os.path.join(label_path, img_name)

        img = cv2.imread(img_path)
        if img is None:
            print("FAILED TO READ:", img_path)
            continue

        # ✅ Use ONLY this pipeline
        results, _ = utils.detect_hand(img, detector)
        landmarks  = utils.extract_landmarks(results)
        normalized = utils.normalize_landmarks(landmarks)

        if normalized is not None and len(normalized) == 126:
            X.append(normalized)
            y.append(label)

print("Total samples:", len(X))

X = np.array(X)
y = np.array(y)

print("Feature shape:", X.shape)

Folders: ['L', 'M', 'J', 'K', 'I', 'G', 'H', 'F', 'E', 'D', 'B', 'A', 'C']
L -> 100
M -> 100
J -> 100
K -> 100
I -> 100
G -> 100
H -> 100
F -> 100
E -> 100
D -> 100
B -> 100
A -> 100
C -> 100
Total samples: 1271
Feature shape: (1271, 126)


In [8]:
import cv2

img = cv2.imread("/content/drive/MyDrive/CV_PROJECT/data_set/A/0.jpg")

features = get_hand_landmarks(img)

print(features)

[0.8164389133453369, 0.6110739707946777, -4.4186759851072566e-07, 0.7318445444107056, 0.5582235455513, -0.05518126115202904, 0.6541677713394165, 0.48426008224487305, -0.08833058923482895, 0.5937398076057434, 0.42149975895881653, -0.11284118890762329, 0.5411752462387085, 0.35431376099586487, -0.13349945843219757, 0.7163442969322205, 0.4723316729068756, -0.07934313267469406, 0.6046066880226135, 0.5708255767822266, -0.10834269970655441, 0.6265426874160767, 0.5810996890068054, -0.12539008259773254, 0.6589231491088867, 0.5487363338470459, -0.1406056433916092, 0.7829532623291016, 0.511196494102478, -0.0728580579161644, 0.6488716006278992, 0.6348496675491333, -0.09393486380577087, 0.6826589107513428, 0.6266342401504517, -0.10563142597675323, 0.721065878868103, 0.5926278829574585, -0.1235443502664566, 0.8357852101325989, 0.5628750324249268, -0.06887183338403702, 0.713642954826355, 0.6793268918991089, -0.08746148645877838, 0.7453191876411438, 0.6727267503738403, -0.0869474709033966, 0.785077691

In [9]:
from sklearn.model_selection import train_test_split
import numpy as np

# convert to numpy (important)
X = np.array(X)
y = np.array(y)

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # 🔥 very important
)

# check sizes
print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (1016, 126)
Test: (255, 126)


In [10]:
print("Unique labels in train:", np.unique(y_train))
print("Unique labels in test:", np.unique(y_test))

Unique labels in train: ['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M']
Unique labels in test: ['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'L' 'M']


In [11]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
for k in range(1, 11):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)

    y_pred = knn.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"K={k} → Accuracy={acc:.4f}")

K=1 → Accuracy=0.9922
K=2 → Accuracy=0.9961
K=3 → Accuracy=0.9961
K=4 → Accuracy=0.9961
K=5 → Accuracy=0.9961
K=6 → Accuracy=0.9961
K=7 → Accuracy=0.9961
K=8 → Accuracy=0.9961
K=9 → Accuracy=0.9961
K=10 → Accuracy=0.9961


In [12]:
knn = KNeighborsClassifier(n_neighbors=3)

knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("KNN Accuracy:", acc)

KNN Accuracy: 0.996078431372549


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("Random Forest Accuracy:", acc)

Random Forest Accuracy: 0.9921568627450981


In [14]:
for n in [50, 100, 150, 200]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"Trees={n} → Accuracy={acc:.4f}")


Trees=50 → Accuracy=0.9922
Trees=100 → Accuracy=0.9922
Trees=150 → Accuracy=0.9922
Trees=200 → Accuracy=0.9922


In [15]:
for n in [50, 100, 150, 200]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"Trees={n} → Accuracy={acc:.4f}")

Trees=50 → Accuracy=0.9922
Trees=100 → Accuracy=0.9922
Trees=150 → Accuracy=0.9922
Trees=200 → Accuracy=0.9922


In [16]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

svm = SVC()

svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("SVM Accuracy:", acc)

SVM Accuracy: 0.9137254901960784


In [17]:
kernels = ['linear', 'rbf', 'poly']

for k in kernels:
    svm = SVC(kernel=k)
    svm.fit(X_train, y_train)

    y_pred = svm.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"Kernel={k} → Accuracy={acc:.4f}")

Kernel=linear → Accuracy=0.9804
Kernel=rbf → Accuracy=0.9137
Kernel=poly → Accuracy=0.9176


In [18]:
best_model = RandomForestClassifier(n_estimators=100, random_state=42)
best_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [19]:
import pickle

with open("/content/drive/MyDrive/CV_PROJECT/model/isl_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Final model saved!")

Final model saved!
